# 07 — Interpretabilite : Grad-CAM + SHAP

**Objectif (exigence jury 3)** : interpretabilite poussee des modeles via **Grad-CAM** (sur les 3 modeles, par classe et sur des cas mal classes) et **SHAP** (sur le meilleur modele).

Question cle : *le modele regarde-t-il les champs pulmonaires, ou des artefacts (marqueurs texte, bords) ?*

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# --- Portabilite Colab / local ---
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Montez votre Drive et placez-y le projet, puis ajustez ce chemin.
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/covid_19_radiographie"  # <-- a adapter
    os.environ["COVID_BASE_DIR"] = PROJECT_DIR
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)
    # Installe les dependances manquantes sur Colab
    !pip -q install shap
else:
    # En local : racine = parent du dossier notebooks/
    PROJECT_DIR = os.path.abspath("..")
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)

from src.utils.env import setup_environment, ensure_dirs, get_paths
INFO = setup_environment()
PATHS = ensure_dirs()
print("Projet :", PROJECT_DIR)

In [ ]:
import numpy as np, json
import tensorflow as tf
from src.data import tf_pipeline as tp
from src.models import transfer_learning as tl
from src.visualization import interpretability as interp

BACKBONES = ['efficientnetb0', 'resnet50', 'inceptionv3']
models = {}
for b in BACKBONES:
    path = PATHS['models_transfer'] / f'{b}_best.keras'
    if path.exists():
        models[b] = tf.keras.models.load_model(path, compile=False)
        print('charge :', b)
    else:
        print('MANQUANT (lancez le notebook 06) :', path)

## 1. Grad-CAM — un exemple correct par classe (chaque modele)

Pour chaque modele et chaque classe, on selectionne une image de test correctement classee et on affiche `originale | heatmap | superposition`.

In [ ]:
import pandas as pd
test_df = tp.load_split_df('test')

def one_correct_per_class(model, spec, max_try=40):
    picks = {}
    for cls in tp.CLASS_NAMES:
        sub = test_df[test_df['class'] == cls].head(max_try)
        for _, row in sub.iterrows():
            raw = interp.load_raw_rgb(row['filepath'], spec.img_size)
            batch = spec.preprocess_fn(np.expand_dims(raw.astype('float32'), 0))
            pred = int(model.predict(batch, verbose=0).argmax())
            if tp.CLASS_NAMES[pred] == cls:
                picks[cls] = row['filepath']; break
    return picks

for b, model in models.items():
    spec = tl.get_spec(b)
    picks = one_correct_per_class(model, spec)
    for cls, fp in picks.items():
        raw = interp.load_raw_rgb(fp, spec.img_size)
        interp.gradcam_panel_for_image(
            model, raw, spec.preprocess_fn, spec.last_conv_layer,
            save_path=PATHS['figures'] / f'gradcam_{b}_{cls.replace(" ", "_")}.png',
            true_label=cls)

## 2. Grad-CAM — cas mal classes

Analyse des erreurs : on visualise des images mal classees pour comprendre **ou** le modele regarde quand il se trompe (confusions attendues COVID <-> Lung_Opacity).

In [ ]:
def misclassified_examples(model, spec, n=4, max_scan=300):
    found = []
    for _, row in test_df.sample(frac=1, random_state=42).head(max_scan).iterrows():
        raw = interp.load_raw_rgb(row['filepath'], spec.img_size)
        batch = spec.preprocess_fn(np.expand_dims(raw.astype('float32'), 0))
        pred = int(model.predict(batch, verbose=0).argmax())
        if tp.CLASS_NAMES[pred] != row['class']:
            found.append((row['filepath'], row['class']))
        if len(found) >= n: break
    return found

# Sur le meilleur modele (par defaut efficientnetb0 si present)
best = next(iter(models))
spec = tl.get_spec(best)
for fp, true_cls in misclassified_examples(models[best], spec):
    raw = interp.load_raw_rgb(fp, spec.img_size)
    interp.gradcam_panel_for_image(models[best], raw, spec.preprocess_fn,
        spec.last_conv_layer, true_label=true_cls,
        save_path=PATHS['figures'] / f'gradcam_{best}_misclassified_{Path(fp).stem}.png')

## 3. SHAP — meilleur modele (selon F1-macro)

`shap.GradientExplainer` (robuste sur Keras fonctionnel moderne). Background ~50 images train, explication de ~8 images test (2/classe). Usage **qualitatif** (couteux sur images). On choisit le meilleur modele d'apres les metriques du notebook 06.

In [ ]:
# Determination du meilleur modele a partir des metriques sauvegardees
best_model_name, best_f1 = None, -1
for b in models:
    mp = PATHS['metrics'] / f'{b}_test_metrics.json'
    if mp.exists():
        f1 = json.load(open(mp))['f1_macro']
        if f1 > best_f1: best_f1, best_model_name = f1, b
best_model_name = best_model_name or next(iter(models))
print('Meilleur modele pour SHAP :', best_model_name, '(F1-macro=%.3f)' % best_f1)
spec = tl.get_spec(best_model_name)
model = models[best_model_name]

In [ ]:
# Construction des batches background (train) et echantillons (test, 2/classe)
bg_df = tp.load_split_df('train').groupby('class').head(13)   # ~52 images
sample_df = test_df.groupby('class').head(2)                  # 8 images

def to_batch(df):
    imgs = [spec.preprocess_fn(interp.load_raw_rgb(fp, spec.img_size).astype('float32'))
            for fp in df['filepath']]
    return np.stack(imgs)

background = to_batch(bg_df)
samples = to_batch(sample_df)
print('background', background.shape, '| samples', samples.shape)
shap_values = interp.shap_gradient_explain(
    model, background, samples,
    save_path=PATHS['figures'] / f'shap_{best_model_name}.png')

## 4. Lecture / discussion

A commenter dans le rapport : Grad-CAM et SHAP doivent montrer une activation **localisee sur les champs pulmonaires**. Une activation sur les bords, marqueurs ou annotations signalerait un biais (apprentissage de raccourcis), limite connue de ce dataset (biais source-hopital).